Bibliotecas necessárias.

In [1]:
# Instalar ou atualizar biblioteca necessária.
!pip install -q -U datasets

# Importar bibliotecas necessárias
import pandas as pd
import re
import gc
from google.colab import userdata
from datasets import load_dataset

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.9 MB/s eta 0:00:00


Funções auxiliares.

In [2]:
# Função para remover a parte numérica inicial e os possíveis sublinhado que a segue.
def clean_category(category):
  # Dicionário de correções de termos jurídicos.
  corrections_dict = {
      'tributario': 'tributário'
  }
  cleaned_numeric_part = re.sub(r'^\d+_?', '', category)

  # Substituir todos os sublinhados por espaços
  final_string = cleaned_numeric_part.replace('_', ' ')

  # Aplicar correções.
  words = final_string.split()
  corrected_words = []
  for word in words:
    # Verifica se a palavra está no dicionário de correções e aplica
    corrected_words.append(corrections_dict.get(word, word))
  final_string = ' '.join(corrected_words)

  # Converter para modo título: primeira letra de cada palavra maiúscula, as demais minúsculas,
  # com as exceções de alguns artigos e preosições.
  lowercase_exceptions = {'e', 'o', 'a', 'de', 'do', 'dos', 'da', 'das'}
  processed_words = []
  for word in final_string.split():
    if word.lower() in lowercase_exceptions:
      processed_words.append(word.lower())
    else:
      processed_words.append(word.title())
  # Retorno.
  return ' '.join(processed_words)

# Função que recebe a url e retorna um Dataframe.
def load_dataframe(url):
   return pd.read_json(url, lines=True)

# Função recuperar todas as categorias de um Dataframe de forma distinta.
def get_unique_categories(df):
  return df['category'].unique()

# Função para juntar dois Dataframes por um campo chave e opcionalmente selecionar colunas.
def merge_dataframes(df1, df2, key, columns=None):
  merged_df = pd.merge(df1, df2, on=key, how='inner')
  if columns is not None:
    return merged_df[columns]
  return merged_df

Funções que povoam os Dataframes com as questões e respostas disponíveis no github.

In [3]:
# Povoar dataframe de questões do huggin face.
def load_questions(hugging_face_id):
  # Carrega o dataset (geralmente ele vem dividido em 'train', 'test', etc.)
  ds = load_dataset(hugging_face_id)

  # Converte uma partição específica 'train' para um dataframe.
  df = pd.DataFrame(ds['train'])

  # Inserir uma coluna para enumerar as questões, com contagem a partir do número 1, uma vez que a contagem de linhas do python é a partir do 0.
  df.insert(0, 'question', range(1, len(df)+1))

  # Realizar limpeza no campo das categorias.
  #df_questions['category'] = df_questions['category'].apply(clean_category)

  # retorno da Dataframe.
  return df

# Dataframe guidelines.
def load_guidelines():
  # URL das respostas de referência.
  url_guidelines = "https://raw.githubusercontent.com/maritaca-ai/oab-bench/refs/heads/main/data/oab_bench/reference_answer/guidelines.jsonl"
  # Carregar um Dataframe com as respostas a partir do arquivo .jsonl.
  return load_dataframe(url_guidelines)

# Dataframe da junção dos DataFrames df_questions e df_guidelines em um único DataFrame usando o question_id de ambos como atributo da interseção.
# Neste Dataframe dá para ver a pergunta e a resposta da linha guia juntas.
# E, também personalizada as colunas que serão apresentadas.
def load_questions_and_guidelines():
  # Carregar os Dataframes.
  df_questions = load_questions()
  df_guidelines = load_guidelines()

  # Chave do inner join
  key = 'question_id'
  # Colunas selecionar para projeção no Dataframe.
  columns=['question', 'question_id', 'category', 'statement', 'turns', 'system', 'choices']

  # Junção e retorno do Dataframe
  return merge_dataframes(df_questions, df_guidelines, key, columns)

# Dataframe com um subconjunto das perguntas e linhas guias.
# O índice do Dataframe começa do 0 (zero), portanto, a linha 1 é a 0, a 2 é a 1, etc.
# iloc seleciona um intervalo fechado à esquerda e aberto à direita
def load_my_questions_and_guidelines(question_min, question_max):
  # Remover 1 und dando o descontando, uma vez que o valor passado é o número da questão para o ser humano
  # e não a contagem do python.
  question_min -= 1
  df_my_questions = load_questions_and_guidelines().iloc[question_min:question_max].copy()

  # Realizar limpeza das informações da coluna turns, pois é um array.
  # E, só preciso da infomação, quando houver, da primeira posição.
  df_my_questions['turns'] = df_my_questions['turns'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else '')

  # Limpeza também a coluna choices, pois é um dicionário que só preciso do campo turns.
  df_my_questions['choices'] = df_my_questions['choices'].apply(lambda x: x[0]['turns'][0] if isinstance(x, list) and len(x) > 0 and 'turns' in x[0] and isinstance(x[0]['turns'], list) and len(x[0]['turns']) > 0 else '')

  # Retornar dataframe ajustado.
  return df_my_questions

# Excluir Dataframes desnecessários.
def delete_obsoletes_objects():
  # Excluir Dataframes que não são mais necessários, apenas se existirem no escopo global.
  if 'df_questions' in globals():
    del df_questions
  if 'df_guidelines' in globals():
    del df_guidelines
  gc.collect()
# Dataframe guidelines.
def load_guidelines():
  # URL das respostas de referência.
  url_guidelines = "https://raw.githubusercontent.com/maritaca-ai/oab-bench/refs/heads/main/data/oab_bench/reference_answer/guidelines.jsonl"
  # Carregar um Dataframe com as respostas a partir do arquivo .jsonl.
  return load_dataframe(url_guidelines)

# Dataframe da junção dos DataFrames df_questions e df_guidelines em um único DataFrame usando o question_id de ambos como atributo da interseção.
# Neste Dataframe dá para ver a pergunta e a resposta da linha guia juntas.
# E, também personalizada as colunas que serão apresentadas.
def load_questions_and_guidelines():
  # Carregar os Dataframes.
  df_questions = load_questions()
  df_guidelines = load_guidelines()

  # Chave do inner join
  key = 'question_id'
  # Colunas selecionar para projeção no Dataframe.
  columns=['question', 'question_id', 'category', 'statement', 'turns', 'system', 'choices']

  # Junção e retorno do Dataframe
  return merge_dataframes(df_questions, df_guidelines, key, columns)

# Dataframe com um subconjunto das perguntas e linhas guias.
# O índice do Dataframe começa do 0 (zero), portanto, a linha 1 é a 0, a 2 é a 1, etc.
# iloc seleciona um intervalo fechado à esquerda e aberto à direita
def load_my_questions_and_guidelines(question_min, question_max):
  # Remover 1 und dando o descontando, uma vez que o valor passado é o número da questão para o ser humano
  # e não a contagem do python.
  question_min -= 1
  df_my_questions = load_questions_and_guidelines().iloc[question_min:question_max].copy()

  # Realizar limpeza das informações da coluna turns, pois é um array.
  # E, só preciso da infomação, quando houver, da primeira posição.
  df_my_questions['turns'] = df_my_questions['turns'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else '')

  # Limpeza também a coluna choices, pois é um dicionário que só preciso do campo turns.
  df_my_questions['choices'] = df_my_questions['choices'].apply(lambda x: x[0]['turns'][0] if isinstance(x, list) and len(x) > 0 and 'turns' in x[0] and isinstance(x[0]['turns'], list) and len(x[0]['turns']) > 0 else '')

  # Retornar dataframe ajustado.
  return df_my_questions

# Excluir Dataframes desnecessários.
def delete_obsoletes_objects():
  # Excluir Dataframes que não são mais necessários, apenas se existirem no escopo global.
  if 'df_questions' in globals():
    del df_questions
  if 'df_guidelines' in globals():
    del df_guidelines
  gc.collect()

Respostas de referências às questões anteriores e disponíveis no mesmo repositório do github.

In [4]:
# Meu sub-grupo de questões e respostas.
# Subconjunto das questões de 31 a 40.
#question_min = 31
#question_max = 40
#df_my_questions = load_my_questions_and_guidelines(question_min, question_max)

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
dataset = load_dataset(dataset_id, use_auth_token=True)

hugging_face_id = 'eduagarcia/oab_exams'
df_close_questions = load_questions(hugging_face_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/649 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.26M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2210 [00:00<?, ? examples/s]

ValueError: cannot insert question, already exists

Função para definir o nível de complexidade das questões.

In [ ]:
# Classificar os níveis das perguntas.
def classify_difficulty_level():
  # Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
  # Tal chave previamente criada no console do Groq, site: consele.groq.com.
  groq_api_key = userdata.get('groq_api_key')

  # Inicializar o cliente da API.
  client_ai = OpenAI(
      base_url="https://api.groq.com/openai/v1",
      api_key=groq_api_key
  )

  # Modelo llama 3 de 70 bilhões de parâmetros.
  model_id = 'llama-3.3-70b-versatile'

  # Inicializar a coluna 'level'.
  df_my_questions['level'] = None

  # Repetição para percorrer todo Dataframe.
  for index, row in df_my_questions.iterrows():
      # preenchimento dos parâmetros da pergunta, com base na linha corrente.
      numero = row['question']
      contexto = row['statement']
      questao = row['turns']

      # Verificar se a questão está vazia antes de submeter à IA
      if not questao.strip(): # Usar .strip() para considerar strings com apenas espaços como vazias
          response = '1' # Atribui '1' se a questão estiver vazia.
      else:
          # Preparação do prompt em Markdown.
          prompt_usuario = f"""
          Analise a complexidade jurídica da seguinte questão e atribua um nível de dificuldade de 1 a 4, onde:
          1: Estagiário (leis e conceitos jurídicos muito básicos, aplicação direta de uma única regra).
          2: Analista (leis e conceitos jurídicos moderados, aplicação de 1-2 conceitos principais, requer identificação de normas).
          3: Juiz (leis e conceitos jurídicos complexos, múltiplos conceitos interconectados, exige análise aprofundada, interpretação de leis e jurisprudência).
          4: Ministro (leis e conceitos jurídicos muito complexos e intrincados, temas controversos, requer alta interpretação constitucional ou legal, análise de precedentes e doutrina avançada).

          ### Contexto:
          {contexto}

          ### Questão:
          {questao}

          Responda apenas com o número do nível de dificuldade (1, 2, 3 ou 4), sem texto adicional.
          """
          completion = client_ai.chat.completions.create(
              model= model_id,
              messages=[
                # Por questões de sintaxes informo a role, pois é um campo obrigatório,
                # porém o conteúdo é somente o do Markdown.
                {"role": "user", "content": prompt_usuario}
              ],
              temperature=0.1 # Conservador.
          )
          response = completion.choices[0].message.content

      # Armazenar o resultado diretamente no DataFrame df_my_questions
      df_my_questions.loc[index, 'level'] = response

  # Fechar conexão com a IA (somente se você a usou ativamente)
  client_ai.close()

Chamar a função que adiciona o nível de difuculdade no dataframe df_my_questions.

In [ ]:
classify_difficulty_level()

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kn7rg2zeee3t7f7drnrps2kb` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99916, Requested 712. Please try again in 9m2.592s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Liberar espeço não necessário em memória.

In [ ]:
delete_obsoletes_objects()